# NFF-MTL on Steam — corrected pipeline, Colab driver (v0.3.1)

The notebook only drives the CLI. **Every command raises on failure, so "Run all" stops at the first error.**
Each command prints `RUN_DIR=...`; the notebook passes those exact folders on and never globs.

| Stage | Config | Reads test-year performance | Purpose |
| --- | --- | --- | --- |
| probe (`pilot`) | `configs/probe.yaml` | no | 300 batches, 2 000 validation users: seconds per batch, peak GPU memory |
| pilot | `configs/pilot.yaml` | no | full data, 2 epochs: real epoch and evaluation cost |
| tuning | `configs/default.yaml` | no | equal-budget search on 2021 → `final_selected.yaml` |
| final | `final_selected.yaml` | once per model identity | retrain with frozen settings, `evaluate` on 2022 |

Test-year *rows* are read for descriptive counts (data card) in every stage; test-year *scores* only in `final`.
The test registry next to the data cache is a guard against accidental repeated evaluation, not a tamper-proof lock.

**First run: "Run all" stops after the probe by design** (`PROCEED_TO_PILOT = False`), and again before tuning
(`PROCEED_TO_TUNE = False`). Change a switch only after reading the numbers printed above it.

In [ ]:
import json, subprocess, sys
from pathlib import Path

def run(*args):
    """Run a command, stream its output, raise CalledProcessError on a non-zero exit; return RUN_DIR if printed."""
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    run_dir = None
    for line in proc.stdout:
        print(line, end='')
        if line.startswith('RUN_DIR='):
            run_dir = line.strip().split('=', 1)[1]
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, args)
    return run_dir

def cli(*args):
    return run(sys.executable, '-m', 'steam_nffmtl.cli', *args)

def run_dirs(*args):
    """For multi-seed `train`: return every RUN_DIR line printed by the command."""
    out = subprocess.run([sys.executable, '-m', 'steam_nffmtl.cli', *map(str, args)], capture_output=True, text=True)
    print(out.stdout[-4000:], out.stderr[-4000:])
    if out.returncode != 0:
        raise subprocess.CalledProcessError(out.returncode, args)
    return [l.split('=', 1)[1] for l in out.stdout.splitlines() if l.startswith('RUN_DIR=')]

## 0. Setup (A100, High-RAM runtime)
Upload `steam-nffmtl-v0.3.1.zip` to `MyDrive/steamRS/` next to `recommendations.csv` and `games_metadata.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/steamRS')
run('unzip', '-q', '-o', DATA_DIR / 'steam-nffmtl-v0.3.1.zip', '-d', '/content/')
REPO = Path('/content/steam-nffmtl-v0.3.1')
%cd {REPO}
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt')
run(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]')
sys.path.insert(0, str(REPO / 'src'))

In [ ]:
cli('preflight', '--require-cuda', '--out', DATA_DIR / 'preflight')

## 1. Tests (GPU tests included; skipped/failed tests are listed by `-rs`). Stop if anything fails.

In [ ]:
run(sys.executable, '-m', 'pytest', '-q', '-rs')

## 2. Prepare the data
Cache folder named by the preprocessing settings; size/mtime checked on every load, SHA-256 re-computed when they differ (always in `final`, `verify_source_hash: true`).

In [ ]:
prep = Path(cli('prepare', '--config', 'configs/default.yaml'))
card = json.loads((prep / 'data_card.json').read_text())
print(json.dumps({k: card[k] for k in ['raw', 'duplicates_removed', 'repeated_pairs_across_partitions', 'kcore', 'train', 'val', 'test', 'tags', 'unverified_assumptions']}, indent=1))
cli('estimate', '--config', 'configs/default.yaml', '--variant', 'full')

## 3a. Probe: limited training batches (first GPU run)
The batch limit shortens NFF-MTL training only. Data loading, the training tensors on the GPU and the validation
pass use the full prepared data; baseline fits in the next cell also use the full training data (with few BPR/ALS iterations).

In [ ]:
probe = Path(cli('train', '--config', 'configs/probe.yaml', '--variant', 'full', '--seeds', '0'))
h = json.loads((probe / 'training.json').read_text())['history'][0]
print({k: h[k] for k in ['batches', 'full_epoch_batches', 'seconds_per_batch', 'val_seconds', 'val_users', 'peak_gpu_memory_gb', 'host_max_rss_gb']})
print('extrapolated full epoch (min):', round(h['seconds_per_batch'] * h['full_epoch_batches'] / 60, 1))

Baseline probe (optional; ItemKNN-CF and TagKNN fit on the full training data, BPR 5 / ALS 2 iterations).

In [ ]:
probe_base = Path(cli('baselines', '--config', 'configs/probe.yaml', '--split', 'val', '--seed', '0'))
for f in sorted((probe_base / 'results').glob('*__val-positive.json')):
    m = json.loads(f.read_text()); print(m['model'], 'fit s:', m.get('fit_seconds'), 'eval s:', m['coverage']['seconds'])

In [ ]:
# ---- Gate 1. Read the probe numbers above (seconds per batch, extrapolated epoch, peak GPU memory, host RSS,
# baseline fit/eval seconds). Only then set this to True and run the cells below.
PROCEED_TO_PILOT = False
if not PROCEED_TO_PILOT:
    raise RuntimeError('Stopped after the probe by design. Review the probe output, then set PROCEED_TO_PILOT = True.')

## 3b. Pilot: full data, 2 epochs (run only if the probe extrapolation is acceptable)

In [ ]:
pilot_train = Path(cli('train', '--config', 'configs/pilot.yaml', '--variant', 'full', '--seeds', '0'))
print([(e['epoch'], e['train_seconds'], e['val_seconds'], e['peak_gpu_memory_gb']) for e in json.loads((pilot_train / 'training.json').read_text())['history']])

**Decide here.** Tuning runs `tuning.n_trials` trials for every model (`budget_policy: equal`; a smaller grid stops
`tune` before any trial). Equal trial counts are not equal compute time: one NFF-MTL trial is a full training run.
Final cost ≈ 15 NFF-MTL trainings (3 variants × 5 seeds) plus test evaluations. Lower `tuning.n_trials` with `--set`
if needed; it stays equal across models and is recorded in `selection.json`. Only models that were actually run may
be reported (EASE is off by default; NCF and LightGCN are not implemented).

## 4. Tuning on the validation year (equal budget)

In [ ]:
# ---- Gate 2. Decide the tuning budget from the pilot (see the note above) before running this cell.
PROCEED_TO_TUNE = False
INCLUDE_EASE = False   # enable only if `estimate` shows that EASE's peak memory fits this runtime
if not PROCEED_TO_TUNE:
    raise RuntimeError('Stopped before tuning by design. Set PROCEED_TO_TUNE = True after reviewing the pilot.')
models = ['tagknn', 'itemknn', 'bpr', 'als', 'nffmtl'] + (['ease'] if INCLUDE_EASE else [])
tune_args = ['tune', '--config', 'configs/default.yaml', '--models', *models]
if INCLUDE_EASE:
    tune_args += ['--set', 'baselines.ease.enabled=true']
tune = Path(cli(*tune_args))
print((tune / 'selection.json').read_text())   # trials_used, grid_size, budget_policy per model
FINAL_CFG = tune / 'final_selected.yaml'       # stage final, selected hyperparameters, strict source hashing

## 5. Final: retrain with frozen settings, evaluate each checkpoint once on 2022, compare

In [ ]:
final_runs = []
# final_runs.append(cli('baselines', '--config', FINAL_CFG, '--split', 'test', '--seed', 0))
# for s in [1, 2, 3, 4]:   # seed-dependent baselines only
#     final_runs.append(cli('baselines', '--config', FINAL_CFG, '--split', 'test', '--seed', s, '--only', 'Random', 'BPR-MF', 'ALS'))
# for variant in ['full', 'no_content', 'single_task']:
#     for train_dir in run_dirs('train', '--config', FINAL_CFG, '--variant', variant, '--seeds', 0, 1, 2, 3, 4):
#         final_runs.append(cli('evaluate', '--config', FINAL_CFG, '--split', 'test', '--run', train_dir))
# cmp = Path(cli('compare', '--config', FINAL_CFG, '--reference', 'NFF-MTL', '--results', *final_runs))
# import pandas as pd; pd.read_csv(cmp / 'comparison_table.csv', index_col=0).filter(regex='_mean$|runs').round(4)